# Profile time

Is it better to use the Jacobian or the jvp/vjp with vectors instead? Need to profile with differing sizes too. 

## Imports and model


In [1]:
from timeit import timeit

import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
from typing import Sequence, List, Tuple, Callable
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import itertools
from jax.scipy.linalg import block_diag
import optax
from datasets import load_dataset
from functools import partial


from jax import random
# Seeding for random operations
main_rng = random.PRNGKey(42)

# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


In [2]:
def img_to_patch(x, patch_size, flatten_channels=True):
    """
    Inputs:
        x - torch.Tensor representing the image of shape [B, H, W, C]
        patch_size - Number of pixels per dimension of the patches (integer)
        flatten_channels - If True, the patches will be returned in a flattened format
                           as a feature vector instead of a image grid.
    """
    B, H, W, C = x.shape
    x = x.reshape(B, H//patch_size, patch_size, W//patch_size, patch_size, C)
    x = x.transpose(0, 1, 3, 2, 4, 5)    # [B, H', W', p_H, p_W, C]
    x = x.reshape(B, -1, *x.shape[3:])   # [B, H'*W', p_H, p_W, C]
    if flatten_channels:
        x = x.reshape(B, x.shape[1], -1) # [B, H'*W', p_H*p_W*C]
    return x

In [3]:
class AttentionBlock(nn.Module):
    embed_dim : int   # Dimensionality of input and attention feature vectors
    hidden_dim : int  # Dimensionality of hidden layer in feed-forward network
    num_heads : int   # Number of heads to use in the Multi-Head Attention block
    # dropout_prob : float = 0.0  # Amount of dropout to apply in the feed-forward network

    def setup(self):
        self.attn = nn.MultiHeadDotProductAttention(num_heads=self.num_heads)
        self.linear = [
            nn.Dense(self.hidden_dim),
            nn.gelu,
            # nn.Dropout(self.dropout_prob),
            nn.Dense(self.embed_dim)
        ]
        self.layer_norm_1 = nn.LayerNorm()
        self.layer_norm_2 = nn.LayerNorm()
        # self.dropout = nn.Dropout(self.dropout_prob)

    def __call__(self, x):
        inp_x = self.layer_norm_1(x)
        attn_out = self.attn(inputs_q=inp_x, inputs_kv=inp_x)
        # x = x + self.dropout(attn_out, deterministic=not train)
        x = x + attn_out

        linear_out = self.layer_norm_2(x)
        for l in self.linear:
            linear_out = l(linear_out)
        # x = x + self.dropout(linear_out, deterministic=not train)
        x = x + linear_out
        
        return x

In [4]:
class Preprocessor(nn.Module):
    embed_dim: int
    patch_size: int
    num_patches: int

    def setup(self):
        self.input_layer = nn.Dense(self.embed_dim)
        self.pos_embedding = self.param('pos_embedding',
                                        nn.initializers.normal(stddev=1.0),
                                        (1, 1 + self.num_patches, self.embed_dim))
        self.cls_token = self.param('cls_token',
                                    nn.initializers.normal(stddev=1.0),
                                    (1, 1, self.embed_dim))

    def __call__(self, x):
        # Preprocess input
        x = img_to_patch(x, self.patch_size)
        B, T, _ = x.shape
        x = self.input_layer(x)

        # Add CLS token and positional encoding
        cls_token = self.cls_token.repeat(B, axis=0)
        x = jnp.concatenate([cls_token, x], axis=1)
        x = x + self.pos_embedding[:, :T + 1]
        return x


class ClassificationHead(nn.Module):
    embed_dim: int
    num_classes: int

    def setup(self):
        self.mlp_head = nn.Sequential([
            nn.LayerNorm(),
            nn.Dense(self.num_classes)
        ])

    def __call__(self, x):
        # Perform classification prediction
        cls = x[:, 0]
        out = self.mlp_head(cls)
        return out


class VisionTransformer(nn.Module):
    embed_dim: int     # Dimensionality of input and attention feature vectors
    hidden_dim: int    # Dimensionality of hidden layer in feed-forward network
    num_heads: int     # Number of heads to use in the Multi-Head Attention block
    num_channels: int  # Number of channels of the input (3 for RGB)
    num_layers: int    # Number of layers to use in the Transformer
    num_classes: int   # Number of classes to predict
    patch_size: int    # Number of pixels that the patches have per dimension
    num_patches: int   # Maximum number of patches an image can have
    # dropout_prob: float = 0.0  # Amount of dropout to apply in the feed-forward network

    def setup(self):
        # All layers
        self.layers = [Preprocessor(self.embed_dim, self.patch_size, self.num_patches)] + [
            AttentionBlock(self.embed_dim,
                                           self.hidden_dim,
                                           self.num_heads,
                                           ) for _ in range(self.num_layers)
        ] + [ClassificationHead(self.embed_dim, self.num_classes)]


    def __call__(self, x, layer_out: bool = False) -> Tuple[jax.Array, List[jax.Array]]:
        intermediate_inputs = [x]
        y = x
        
        # Apply Transformer
        for i, layer in enumerate(self.layers):
            y = layer(y)
            y = self.perturb(f'layer_{i+1}', y)
            intermediate_inputs.append(y)
            
        # The final `y` is the final output. The inputs list is one item too long.
        final_output = intermediate_inputs.pop()

        if not layer_out:
            return final_output
            
        return final_output, intermediate_inputs


    def get_jacobian_calculators(self) -> List[Callable]:
        """
        Creates a list of functions that compute the Jacobian of each layer's
        output with respect to its parameters.

        This method is called via `model.apply(..., method=...)`, which ensures
        that `self.layers` is available.

        Adapted for VisualTransformer, noting that the middle layers are the same functions as they're all 
        just self-attention. 
        """
        jacobian_fns = []

        def layer_apply_fn(layer_instance: nn.Module, params: dict, layer_input: jax.Array) -> jax.Array:
            """A wrapper to call a single layer's apply method."""
            return layer_instance.apply({'params': params}, layer_input)

        # Compute Jacobian for the preprocessor layer
        # jacobian_fns.append(jax.jacrev(partial(layer_apply_fn, self.layers[0]), argnums=0))
        jacobian_fns.append(jax.jacfwd(partial(layer_apply_fn, self.layers[0]), argnums=0))
    
        # Compute Jacobian for one representative attention block
        attention_block = self.layers[1]  # Assuming all middle layers are identical
        # attention_jac_fn = jax.jacrev(partial(layer_apply_fn, attention_block), argnums=0)
        attention_jac_fn = jax.jacfwd(partial(layer_apply_fn, attention_block), argnums=0)
    
        # Reuse the Jacobian function for all identical attention blocks
        jacobian_fns.extend([attention_jac_fn] * self.num_layers)
    
        # Compute Jacobian for the classification head
        # jacobian_fns.append(jax.jacrev(partial(layer_apply_fn, self.layers[-1]), argnums=0))
        jacobian_fns.append(jax.jacfwd(partial(layer_apply_fn, self.layers[-1]), argnums=0))

        return jacobian_fns



# Create scenarios and time 

In [30]:
key = jax.random.PRNGKey(0)
model =  VisionTransformer(embed_dim=8, # Make these numbers small for testing first
                              hidden_dim=32,
                              num_heads=4,
                              num_channels=1,
                              num_layers=4,
                              num_classes=10,
                              patch_size=4,
                              num_patches=49,
                             )

# Hmm, input must match batch size? 
bsz = 64
params = model.init(key, jnp.ones((bsz, 28, 28, 1)))
print(sum(x.size for x in jax.tree.leaves(params['params'])))
batch_X =  jnp.ones((bsz, 28, 28, 1))
logits, layer_vals  = model.apply(params, batch_X, layer_out=True)


def explicit_global_jacobian(params, batch_X):
    jacobian = jax.jacrev(
        lambda params, perturbations, batch_X: model.apply(
            {'params': params, 'perturbations': perturbations}, batch_X
        ),
        argnums=1
    )(
        params['params'], params['perturbations'], batch_X 
    )
    flattened, _ = jax.tree.flatten(
        jacobian
    )
    global_adjoints = [leaf.reshape((leaf.shape[0] * leaf.shape[1], -1)) for leaf in flattened]
    
    return global_adjoints
global_adjoints = explicit_global_jacobian(params, batch_X)

4138


## Time the layer stuff

In [33]:
layer_jacobian = model.apply(
    {'params': params['params']}, method=model.get_jacobian_calculators
)


# Now do the layer calculations
@jax.jit
def get_layer_jacs_mat(params, layer_vals):
    layers = []
    for i, (jac_fn, layer_input) in enumerate(zip(layer_jacobian, layer_vals)):
        layer_params = params['params'][f'layers_{i}']
        jacobian_pytree = jac_fn(layer_params, layer_input)            
        layer_weights = jax.tree.flatten(jacobian_pytree)[0]
        layer_weights = [weights.reshape((global_adjoints[i].shape[1], -1)) for weights in layer_weights]
        layer_weights = jnp.concatenate(layer_weights, axis=-1)
        layers.append(layer_weights)
    
    return layers

In [35]:
%timeit get_layer_jacs_mat(params, layer_vals)

149 μs ± 33.9 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


## First, we want to time the case of creating the full Jacobian for adjoints

In [21]:
# @jax.jit
def explicit_global_jacobian(params, batch_X):
    jacobian = jax.jacrev(
        lambda params, perturbations, batch_X: model.apply(
            {'params': params, 'perturbations': perturbations}, batch_X
        ),
        argnums=1
    )(
        params['params'], params['perturbations'], batch_X 
    )
    flattened, _ = jax.tree.flatten(
        jacobian
    )
    global_adjoints = [leaf.reshape((leaf.shape[0] * leaf.shape[1], -1)) for leaf in flattened]
    
    return global_adjoints

In [15]:
%timeit explicit_global_jacobian(params, jnp.ones((bsz, 28, 28, 1)))

459 μs ± 124 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [42]:
params['perturbations']

{'layer_1': Array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        ...,
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]]], dtype=float64),
 'layer_2': Array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        ...,
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]]], dtype=float64),
 'layer_3': Array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        ...,
 
        [[0., 0., 0., ..., 0., 0., 0.

In [52]:
params['perturbations']

{'layer_1': Array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        ...,
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]]], dtype=float64),
 'layer_2': Array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        ...,
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]]], dtype=float64),
 'layer_3': Array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        ...,
 
        [[0., 0., 0., ..., 0., 0., 0.

In [54]:
def model_apply_fn(perturbations, params, batch_X):
    return model.apply(
        {'params': params, 'perturbations': perturbations}, batch_X
    )

# Get the VJP function. vjp does not have a 'has_aux' argument.
output, vjp_fn = jax.vjp(
    model_apply_fn,
    params['perturbations'],  # The argument you're differentiating wrt
    params['params'],
    batch_X,
)

# Call the VJP function with your vector
vjp_result = vjp_fn(jnp.ones((64,10)))

In [50]:
vjp_result

({'layer_1': Array([[[ 2.07438193e-01, -1.51745165e-01,  2.41372124e+00, ...,
           -1.13305294e+00, -1.03731818e-01,  1.46925956e-01],
          [ 4.82464028e-02,  4.37125333e-03,  1.64986292e-02, ...,
            7.18312783e-02, -4.91143862e-02, -4.83478943e-02],
          [ 1.75125307e-02,  5.28686376e-03,  1.27755019e-02, ...,
            2.54904627e-02, -2.29675095e-02, -1.66406758e-02],
          ...,
          [ 7.75007999e-03, -4.55918690e-03,  1.41412781e-02, ...,
            4.30370016e-02, -3.60464512e-02,  9.98305039e-04],
          [ 6.73069186e-04, -1.23104204e-02,  5.86782611e-03, ...,
            2.03751711e-02, -9.25236983e-03,  8.74948186e-03],
          [ 3.63434696e-02, -1.09339077e-02, -1.29857518e-02, ...,
            3.91037460e-02, -1.48552784e-02, -2.72984916e-02]],
  
         [[ 2.07438193e-01, -1.51745165e-01,  2.41372124e+00, ...,
           -1.13305294e+00, -1.03731818e-01,  1.46925956e-01],
          [ 4.82464028e-02,  4.37125333e-03,  1.64986292e-02, ...,
            7.18312783e-02, -4.91143862e-02, -4.83478943e-02],
          [ 1.75125307e-02,  5.28686376e-03,  1.27755019e-02, ...,
            2.54904627e-02, -2.29675095e-02, -1.66406758e-02],
          ...,
          [ 7.75007999e-03, -4.55918690e-03,  1.41412781e-02, ...,
            4.30370016e-02, -3.60464512e-02,  9.98305039e-04],
          [ 6.73069186e-04, -1.23104204e-02,  5.86782611e-03, ...,
            2.03751711e-02, -9.25236983e-03,  8.74948186e-03],
          [ 3.63434696e-02, -1.09339077e-02, -1.29857518e-02, ...,
            3.91037460e-02, -1.48552784e-02, -2.72984916e-02]],
  
         [[ 2.07438193e-01, -1.51745165e-01,  2.41372124e+00, ...,
           -1.13305294e+00, -1.03731818e-01,  1.46925956e-01],
          [ 4.82464028e-02,  4.37125333e-03,  1.64986292e-02, ...,
            7.18312783e-02, -4.91143862e-02, -4.83478943e-02],
          [ 1.75125307e-02,  5.28686376e-03,  1.27755019e-02, ...,
            2.54904627e-02, -2.29675095e-02, -1.66406758e-02],
          ...,
          [ 7.75007999e-03, -4.55918690e-03,  1.41412781e-02, ...,
            4.30370016e-02, -3.60464512e-02,  9.98305039e-04],
          [ 6.73069186e-04, -1.23104204e-02,  5.86782611e-03, ...,
            2.03751711e-02, -9.25236983e-03,  8.74948186e-03],
          [ 3.63434696e-02, -1.09339077e-02, -1.29857518e-02, ...,
            3.91037460e-02, -1.48552784e-02, -2.72984916e-02]],
  
         ...,
  
         [[ 2.07438193e-01, -1.51745165e-01,  2.41372124e+00, ...,
           -1.13305294e+00, -1.03731818e-01,  1.46925956e-01],
          [ 4.82464028e-02,  4.37125333e-03,  1.64986292e-02, ...,
            7.18312783e-02, -4.91143862e-02, -4.83478943e-02],
          [ 1.75125307e-02,  5.28686376e-03,  1.27755019e-02, ...,
            2.54904627e-02, -2.29675095e-02, -1.66406758e-02],
          ...,
          [ 7.75007999e-03, -4.55918690e-03,  1.41412781e-02, ...,
            4.30370016e-02, -3.60464512e-02,  9.98305039e-04],
          [ 6.73069186e-04, -1.23104204e-02,  5.86782611e-03, ...,
            2.03751711e-02, -9.25236983e-03,  8.74948186e-03],
          [ 3.63434696e-02, -1.09339077e-02, -1.29857518e-02, ...,
            3.91037460e-02, -1.48552784e-02, -2.72984916e-02]],
  
         [[ 2.07438193e-01, -1.51745165e-01,  2.41372124e+00, ...,
           -1.13305294e+00, -1.03731818e-01,  1.46925956e-01],
          [ 4.82464028e-02,  4.37125333e-03,  1.64986292e-02, ...,
            7.18312783e-02, -4.91143862e-02, -4.83478943e-02],
          [ 1.75125307e-02,  5.28686376e-03,  1.27755019e-02, ...,
            2.54904627e-02, -2.29675095e-02, -1.66406758e-02],
          ...,
          [ 7.75007999e-03, -4.55918690e-03,  1.41412781e-02, ...,
            4.30370016e-02, -3.60464512e-02,  9.98305039e-04],
          [ 6.73069186e-04, -1.23104204e-02,  5.86782611e-03, ...,
            2.03751711e-02, -9.25236983e-03,  8.74948186e-03],
          [ 3.63434696e-02, -1.09339077e-02, -1.29857518e-02, ...,
            3.91037460e-02, -1.48552784e-02,